# Ch 15. マルチヘッドアテンション — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: MHA のヘッド数を 1、4、8、16 と変えて学習曲線を比較せよ。

### 解答

公平な比較では $d_{model}$、データ、初期化方針、最適化ステップを固定し、ヘッド数だけを変える。1 回の短い実行で優劣を断定せず、複数シードの平均と分散を報告する。以下の検査は全設定で $h d_k=d_{model}$ を保証する。


In [ ]:
import numpy as np

# Reduced controlled training: only the head count changes; total feature width stays 64.
rng = np.random.default_rng(1501)
X = rng.normal(size=(192, 64))
teacher = rng.normal(size=64)
y = X @ teacher + 0.1 * rng.normal(size=192)
curves = {}
for heads in (1, 4, 8, 16):
    width = 64 // heads
    w = np.zeros((heads, width))
    losses = []
    for _ in range(80):
        pred = np.einsum("nhd,hd->n", X.reshape(192, heads, width), w)
        error = pred - y
        grad = (2 / len(X)) * np.einsum("nhd,n->hd", X.reshape(192, heads, width), error)
        w -= 0.08 * grad
        losses.append(float(np.mean(error**2)))
    curves[heads] = losses
assert all(v[-1] < 0.02 * v[0] for v in curves.values())
print({h: {"initial_mse": round(v[0], 4), "final_mse": round(v[-1], 4)} for h, v in curves.items()})


## 問題 2

**問題**: アテンションヘッドのパターンを可視化し、「positional head」（対角線に集中）が存在するか確認せよ。

### 解答

アテンション行列 $A$ の対角集中度は $\operatorname{mean}_i A_{ii}$ で定量化できる。一様アテンションの基準値は $1/L$ なので、これを十分上回り複数サンプルで再現すれば positional head の根拠になる。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

length = 12
positions = np.arange(length)
scores = np.stack([
    -2.0 * np.abs(positions[:, None] - positions[None, :]),
    -0.2 * np.abs(positions[:, None] - positions[None, :]),
])
attention = np.exp(scores - scores.max(axis=-1, keepdims=True))
attention /= attention.sum(axis=-1, keepdims=True)
diagonal_mass = np.diagonal(attention, axis1=1, axis2=2).mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
for head, axis in enumerate(axes):
    axis.imshow(attention[head], vmin=0, vmax=1, cmap="viridis")
    axis.set_title(f"head {head}: diag={diagonal_mass[head]:.3f}")
plt.close(fig)
assert diagonal_mass[0] > diagonal_mass[1] > 1 / length
print({"diagonal_mass": diagonal_mass.round(4).tolist(), "uniform_baseline": round(1 / length, 4)})


## 問題 3

**問題**: MQA と MHA のパラメータ数の差を計算し、実験で検証せよ。

### 解答

バイアスを除くと MHA の K、V 射影は各 $d^2$、MQA は各 $d(d/h)$ パラメータである。したがって 2 射影で $2d^2(1-1/h)$ を節約し、Q と出力射影は変わらない。


In [ ]:
import numpy as np

d_model, heads = 64, 8
head_dim = d_model // heads
rng = np.random.default_rng(1503)
mha_k = rng.normal(size=(d_model, d_model))
mha_v = rng.normal(size=(d_model, d_model))
mqa_k = rng.normal(size=(d_model, head_dim))
mqa_v = rng.normal(size=(d_model, head_dim))
measured = {"MHA_KV": mha_k.size + mha_v.size, "MQA_KV": mqa_k.size + mqa_v.size}
reference = {"MHA_KV": 2 * d_model**2, "MQA_KV": 2 * d_model * head_dim}
assert measured == reference
print({**measured, "saved_parameters": measured["MHA_KV"] - measured["MQA_KV"],
       "saving_fraction": round(1 - measured["MQA_KV"] / measured["MHA_KV"], 3)})


## 問題 4

**問題**: GQA で $g = 1, 2, 4, 8$ の推論速度とメモリを比較せよ。

### 解答

GQA のキャッシュ要素数は $2Lgd_k$ であり $g$ に線形である。実時間はカーネルとハードウェアに依存するため、以下は決定的なメモリ／帯域代理指標であり、架空の時間測定ではない。


In [ ]:
import time
import numpy as np

# A reduced decode kernel reads every cached K/V element, exposing GQA's bandwidth scaling.
rng = np.random.default_rng(1504)
length, head_dim, repeats = 1024, 32, 80
results = {}
for groups in (1, 2, 4, 8):
    cache = rng.normal(size=(2, length, groups, head_dim)).astype(np.float32)
    start = time.perf_counter()
    checksum = 0.0
    for _ in range(repeats):
        checksum += float(np.sum(cache * 0.0001))
    elapsed = time.perf_counter() - start
    results[groups] = {"bytes": cache.nbytes, "median_proxy_ms": 1000 * elapsed / repeats,
                       "checksum": round(checksum, 4)}
bytes_used = [results[g]["bytes"] for g in (1, 2, 4, 8)]
assert bytes_used == [bytes_used[0] * g for g in (1, 2, 4, 8)]
print({g: {"KiB": r["bytes"] // 1024, "ms_per_read": round(r["median_proxy_ms"], 4)} for g, r in results.items()})


## 問題 5

**問題**: Multi-Head Attention で $W^O$ 行列が必要な理由を説明せよ。

### 解答

$W^O$ は連結された $hd_v$ 次元を $d_{model}$ に写し、ヘッド間情報を学習可能に混合する。これがなければ各ヘッド部分空間は並置されるだけで、残差ストリームとの次元・基底整合は一般に保証されない。


In [ ]:
import numpy as np

rng = np.random.default_rng(1505)
batch, heads, width = 32, 4, 8
head_outputs = rng.normal(size=(batch, heads * width))
target = head_outputs[:, :width] - 0.7 * head_outputs[:, width:2 * width]
# Least-squares W^O learns a cross-head mixture; a fixed concatenation cannot produce this target.
W_o, *_ = np.linalg.lstsq(head_outputs, target, rcond=None)
mixed = head_outputs @ W_o
unmixed = head_outputs[:, :width]
mixed_mse = float(np.mean((mixed - target) ** 2))
unmixed_mse = float(np.mean((unmixed - target) ** 2))
assert mixed_mse < 1e-20 and unmixed_mse > 0.1
print({"learned_WO_mse": mixed_mse, "unmixed_first_head_mse": round(unmixed_mse, 4)})
